In [ ]:
import os 
import zarr
import napari

import numpy as np 
from scipy.ndimage import gaussian_filter, map_coordinates
from scipy.spatial import cKDTree
from skimage.feature import peak_local_max
import matplotlib.pyplot as plt
import dask.array as da

plt.style.use('dark_background')

STEP = 1
Z_RATIO = 4.0
SCALE = (1.0, 4.0, 1.0, 1.0)

VIDEOS = [
    # "44b6_33b596bf", "44b6_40c45f5a", "44b6_551a5dba", # used in baseline (v0)
    # "6bba_61dd1e0d", # used in first iter (v1)
    
    "44b6_87bba6c4",
    "44b6_abf82518", 
    "44b6_c96cfa10",
    "44b6_eb2880fc", 
    "6bba_55c70843",
    "6bba_6ca87370", 
    "6bba_debd7bfa"

    # "6bba_6feeb0b1", # potential r2 - 1338 - rejected
    # "6bba_9e23430b", # potential r2 - 913 - potential r3
    # "6bba_80d12824", # potential r2 - 679 - potential r3
    
    # "6bba_1f58c2f6", # r3 selected

]


In [2]:
def norm(f):
    lo, hi = np.percentile(f, (1, 99.8))
    return np.clip((f - lo) / (hi - lo + 1e-6), 0, 1)

def dog_response(f, sigma_xy=3.0, sigma_z=2.0, k=1.6):   # sigma_z in voxels
    s1 = (sigma_z, sigma_xy, sigma_xy)
    s2 = tuple(k * s for s in s1)
    return gaussian_filter(f, s1) - gaussian_filter(f, s2)

def find_peaks(dog, min_dist=(3, 4, 4), rel_thresh=0.1, merge_dip=0.85, link_r=(8, 8, 8)):
    scale = np.percentile(dog, 99.9)
    
    p = peak_local_max(
        dog, 
        footprint=np.ones([2 * m + 1 for m in min_dist], bool),
        exclude_border=False, 
        threshold_abs=rel_thresh * scale
        )
    
    if len(p) == 0:
        return p,
    
    v = dog[tuple(p.T)]
    o = np.argsort(-v); p, v = p[o], v[o]                    # strongest first
    lr = np.array(link_r, float)
    
    tree = cKDTree(p / lr)
    keep = np.ones(len(p), bool)

    for i in range(len(p)):
        for j in tree.query_ball_point(p[i] / lr, 1.0):
            if j >= i or not keep[j]:
                continue
                
            line = p[j] + np.linspace(0, 1, 16)[:, None] * (p[i] - p[j])
            r = max(map_coordinates(dog, line.T, order=1).min(), 0) / v[i]
            
            if r >= merge_dip:
                keep[i] = False; break
              
    return p[keep]

In [4]:
def bootstrap_points(
        videos,step=STEP, sigma_xy=3.0, sigma_z=2.0,
        min_dist=(3, 4, 4), rel_thresh=0.1, overwrite=False
    ):
    """Saves labels/<vid>_pts.npy   (N,4: t,z,y,x)
             labels/<vid>_score.npy (N,)  seed quality 0..1
             labels/dog_response/<vid>_dog.npy (T,Z,Y,X) float16, zeros on unsampled frames."""
   
    os.makedirs('unet_labels/dog_response', exist_ok=True)
   
    for vid in videos:
        pts_f = f'unet_labels/{vid}_pts.npy'
        dog_f = f'unet_labels/dog_response/{vid}_dog.npy'
   
        need_pts = overwrite or not os.path.exists(pts_f)   # never clobber hand edits
        need_dog = overwrite or not os.path.exists(dog_f)
   
        if not (need_pts or need_dog):
            print(f'{vid}: nothing to do'); continue

        z = zarr.open(f'unet_patches/{vid}.zarr/0')
   
        pts = []
        dogs = (np.lib.format.open_memmap(dog_f, mode='w+', dtype=np.float16, shape=z.shape)
                if need_dog else None)

        for t in range(0, z.shape[0], step):
            f = norm(np.asarray(z[t], dtype=np.float32))
            d = dog_response(f, sigma_xy, sigma_z)
            p = find_peaks(d, min_dist, rel_thresh)
            pts.append(np.c_[np.full(len(p), t), p])
   
            if need_dog:
                dogs[t] = d.astype(np.float16)

        if need_dog:
            dogs.flush(); del dogs
        if need_pts:
            np.save(pts_f, np.concatenate(pts).astype(np.float32))

        print(f'{vid}: {sum(len(p) for p in pts)} seeds, {len(pts)} frames')

In [ ]:
bootstrap_points(VIDEOS)

44b6_33b596bf: 693 seeds, 50 frames
44b6_40c45f5a: 3802 seeds, 200 frames
44b6_551a5dba: 6216 seeds, 228 frames
44b6_87bba6c4: 2999 seeds, 153 frames
44b6_abf82518: 1308 seeds, 96 frames
44b6_c96cfa10: 11886 seeds, 629 frames
44b6_eb2880fc: 7518 seeds, 390 frames
6bba_1f58c2f6: 5414 seeds, 317 frames
6bba_55c70843: 11405 seeds, 490 frames
6bba_61dd1e0d: 2535 seeds, 558 frames
6bba_6ca87370: 10790 seeds, 1081 frames
6bba_6feeb0b1: 11636 seeds, 1389 frames
6bba_80d12824: 5017 seeds, 680 frames
6bba_9e23430b: 4399 seeds, 914 frames
6bba_debd7bfa: 7885 seeds, 766 frames


In [4]:
for vid in VIDEOS:
    out = f'unet_labels/{vid}_pts.npy'
    dog_f = f'unet_labels/dog_response/{vid}_dog.npy'

    video = da.from_zarr(f'unet_patches/{vid}.zarr/0')

    viewer = napari.Viewer(ndisplay=3, title=vid)
    try:
        viewer.axes.visible = viewer.axes.colored = viewer.axes.labels = True
    except AttributeError:
        a = viewer.scene.overlays.axes
        a.visible = a.colored = a.labels = True

    viewer.add_image(
        video, 
        scale=SCALE, 
        colormap='magma', 
        blending='additive', 
        name='video'
    )
    
    if os.path.exists(dog_f):
        dog = np.load(dog_f, mmap_mode='r')
        viewer.add_image(
            dog, 
            scale=SCALE, 
            name='DoG', 
            colormap='viridis',
            blending='additive', 
            visible=False,
            contrast_limits=(0, float(dog[::STEP].max()))
        )
    
    pts = viewer.add_points(
        np.load(out) if os.path.exists(out) else None,
        name='centers', 
        scale=SCALE, 
        ndim=4, 
        size=3, 
        face_color='cyan',
    )

    def save():
        np.save(out, pts.data.astype(np.float32))
        viewer.status = f'saved {len(pts.data)} points'

    viewer.bind_key('s', lambda v: save())

    @viewer.bind_key('g')            # select a point, press G: jump to it in 2D
    def _goto(v):
        sel = list(pts.selected_data)
        if not sel:
            v.status = 'select a point first'; return
        t, z, y, x = pts.data[sel[0]] * SCALE
        v.dims.ndisplay = 2
        v.dims.set_point(0, t); v.dims.set_point(1, z)
        v.camera.center = (0, y, x); v.camera.zoom = 6

    @viewer.bind_key('h')            # toggle disc <-> hbar for selected point(s)
    def _symbol(v):
        sel = list(pts.selected_data)
        if sel:
            sym = np.array([str(s).lower() for s in pts.symbol], dtype=object)
            new = 'hbar' if 'disc' in sym[sel[0]] else 'disc'
            sym[sel] = new; pts.symbol = sym
        else:
            new = 'hbar' if 'disc' in str(pts.current_symbol).lower() else 'disc'
            pts.current_symbol = new

    @pts.mouse_move_callbacks.append  # hover: stored coords in the status bar
    def _hover(layer, event):
        i = layer.get_value(event.position, view_direction=event.view_direction,
                            dims_displayed=event.dims_displayed, world=True)
        if i is not None:
            t, z, y, x = layer.data[i]
            viewer.status = f'pt {i}: t={t:.0f} z={z:.0f} y={y:.0f} x={x:.0f}'

    napari.run()
    save()
    print(f'{vid}: {len(pts.data)} points saved')

could not determine DPI
could not determine DPI


44b6_87bba6c4: 2999 points saved


could not determine DPI


44b6_abf82518: 1308 points saved


could not determine DPI


...

6bba_1f58c2f6: 5414 points saved


could not determine DPI


6bba_55c70843: 11405 points saved


could not determine DPI


6bba_6ca87370: 10790 points saved
6bba_debd7bfa: 7885 points saved
